In [ ]:
# This snippet shows how to serve Bamba in production for high throughput.
# It requires vLLM installed with Mamba support.
# pip install vllm>=0.5.0

from vllm import LLM, SamplingParams

# The model path points to the v1 release on the IBM AI Platform
model_id = "ibm-ai-platform/Bamba-9B-v1"

# We load the model. vLLM automatically detects the hybrid architecture
# and routes the Mamba2 layers to optimized CUDA kernels.
llm = LLM(
    model=model_id,
    trust_remote_code=True,
    max_model_len=4096,
    tensor_parallel_size=1
)

prompts = [
    "The main advantage of a hybrid SSM model is",
    "To optimize a RAG pipeline for an 80GB GPU, you should"
]

sampling_params = SamplingParams(temperature=0.0, max_tokens=100)

# The prefill phase here processes the prompts in parallel.
# Because 29 of the 32 layers are linear, the time to process
# these tokens is significantly lower than a pure transformer.
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt}\nOutput: {generated_text}\n")